# Tufts Dental — Bounding Box Exploration

Explore the Tufts dental dataset: radiograph dimensions, bounding box format,
tooth size distributions, degenerate boxes, and visual samples.

In [ ]:
import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Load the bbox annotations

In [ ]:
BBOX_PATH = "../data/tufts_dental/Segmentation/teeth_bbox.json"
RADIO_DIR = "../data/tufts_dental/Radiographs"

with open(BBOX_PATH) as f:
    data = json.load(f)

print(f"Total annotated images: {len(data)}")
print(f"Sample entry keys: {list(data[0].keys())}")
print(f"First entry External ID: {data[0]['External ID']}")
print(f"First entry tooth count: {len(data[0]['Label']['objects'])}")

## 2. Filename resolution

The JSON uses lowercase `.jpg` but disk files are `.JPG`. Check how many resolve.

In [ ]:
def resolve_image_path(external_id, radio_dir):
    """Case-insensitive file lookup."""
    stem = os.path.splitext(external_id)[0]
    for ext in ['.JPG', '.jpg', '.jpeg', '.png', '.PNG']:
        p = os.path.join(radio_dir, f"{stem}{ext}")
        if os.path.exists(p):
            return p
    return None

resolved = 0
missing = []
for entry in data:
    p = resolve_image_path(entry['External ID'], RADIO_DIR)
    if p:
        resolved += 1
    else:
        missing.append(entry['External ID'])

print(f"Resolved: {resolved}/{len(data)}")
if missing:
    print(f"Missing files: {missing[:10]}{'...' if len(missing) > 10 else ''}")
else:
    print("All files found!")

## 3. Image dimensions

In [ ]:
# Sample a subset to check dimensions (reading all 1000 is slow)
shapes = []
for entry in data[:50]:
    p = resolve_image_path(entry['External ID'], RADIO_DIR)
    if p:
        img = cv2.imread(p)
        shapes.append(img.shape)

unique_shapes = Counter(shapes)
print("Image shape distribution (first 50 images):")
for shape, count in unique_shapes.most_common():
    print(f"  {shape} (HxWxC): {count} images")

## 4. Bounding box format analysis

The bbox format in the JSON is `[top, left, bottom, right]` in absolute pixel coordinates.

Let's verify this and analyze tooth sizes.

In [ ]:
# Gather all bounding box stats
heights = []
widths = []
areas = []
aspect_ratios = []  # h/w
tooth_counts = []
tooth_ids = []
degenerate_count = 0

for entry in data:
    objs = entry['Label']['objects']
    tooth_counts.append(len(objs))
    for o in objs:
        top, left, bottom, right = o['bounding box']
        h = bottom - top
        w = right - left
        heights.append(h)
        widths.append(w)
        areas.append(h * w)
        if w > 0 and h > 0:
            aspect_ratios.append(h / w)
        tooth_ids.append(o['title'])
        if h <= 5 or w <= 5:
            degenerate_count += 1

print(f"Total annotations: {len(heights)}")
print(f"Degenerate boxes (h or w <= 5): {degenerate_count}")
print(f"\nTeeth per image: min={min(tooth_counts)}, max={max(tooth_counts)}, "
      f"mean={np.mean(tooth_counts):.1f}, median={np.median(tooth_counts):.0f}")
print(f"\nBbox heights (px): min={min(heights)}, max={max(heights)}, "
      f"mean={np.mean(heights):.1f}, median={np.median(heights):.0f}")
print(f"Bbox widths (px):  min={min(widths)}, max={max(widths)}, "
      f"mean={np.mean(widths):.1f}, median={np.median(widths):.0f}")
print(f"Bbox areas (px²):  min={min(areas)}, max={max(areas)}, "
      f"mean={np.mean(areas):.0f}, median={np.median(areas):.0f}")

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Filter out degenerate for cleaner histograms
valid_h = [h for h in heights if h > 5]
valid_w = [w for w in widths if w > 5]
valid_areas = [a for a in areas if a > 25]

axes[0, 0].hist(valid_h, bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Bbox Height Distribution')
axes[0, 0].set_xlabel('Height (px)')
axes[0, 0].axvline(np.median(valid_h), color='red', linestyle='--', label=f'median={np.median(valid_h):.0f}')
axes[0, 0].legend()

axes[0, 1].hist(valid_w, bins=50, color='coral', edgecolor='black')
axes[0, 1].set_title('Bbox Width Distribution')
axes[0, 1].set_xlabel('Width (px)')
axes[0, 1].axvline(np.median(valid_w), color='red', linestyle='--', label=f'median={np.median(valid_w):.0f}')
axes[0, 1].legend()

axes[0, 2].hist(aspect_ratios, bins=50, color='seagreen', edgecolor='black')
axes[0, 2].set_title('Aspect Ratio (H/W) Distribution')
axes[0, 2].set_xlabel('H/W ratio')
axes[0, 2].axvline(np.median(aspect_ratios), color='red', linestyle='--', label=f'median={np.median(aspect_ratios):.2f}')
axes[0, 2].legend()

axes[1, 0].hist(tooth_counts, bins=range(0, 35), color='mediumpurple', edgecolor='black')
axes[1, 0].set_title('Teeth Per Image')
axes[1, 0].set_xlabel('Number of teeth')

axes[1, 1].scatter(valid_w, valid_h[:len(valid_w)], alpha=0.1, s=5, color='steelblue')
axes[1, 1].set_title('Width vs Height (valid boxes)')
axes[1, 1].set_xlabel('Width (px)')
axes[1, 1].set_ylabel('Height (px)')
axes[1, 1].plot([0, 250], [0, 250], 'r--', alpha=0.5, label='square')
axes[1, 1].legend()

# Tooth ID distribution (which teeth are most/least annotated)
id_counts = Counter(tooth_ids)
sorted_ids = sorted(id_counts.items(), key=lambda x: int(x[0]))
axes[1, 2].bar([x[0] for x in sorted_ids], [x[1] for x in sorted_ids], color='orange', edgecolor='black')
axes[1, 2].set_title('Annotations Per Tooth ID')
axes[1, 2].set_xlabel('Tooth ID (FDI numbering)')
axes[1, 2].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

## 5. Visualize bounding boxes on radiographs

Draw all tooth bounding boxes on sample images to visually verify the format is correct.

In [ ]:
def draw_bboxes(image_bgr, objects, color=(0, 255, 0), thickness=2):
    """Draw bounding boxes and tooth IDs on image.
    Bbox format: [top, left, bottom, right]."""
    vis = image_bgr.copy()
    for o in objects:
        top, left, bottom, right = o['bounding box']
        h = bottom - top
        w = right - left
        if h <= 5 or w <= 5:  # skip degenerate
            continue
        cv2.rectangle(vis, (left, top), (right, bottom), color, thickness)
        cv2.putText(vis, o['title'], (left, top - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    return vis

In [ ]:
# Show 6 random sample images with bounding boxes
rng = np.random.RandomState(42)
sample_indices = rng.choice(len(data), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()

for ax_idx, idx in enumerate(sample_indices):
    entry = data[idx]
    img_path = resolve_image_path(entry['External ID'], RADIO_DIR)
    img = cv2.imread(img_path)
    objs = entry['Label']['objects']
    
    vis = draw_bboxes(img, objs)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    
    n_valid = sum(1 for o in objs if (o['bounding box'][2] - o['bounding box'][0]) > 5
                  and (o['bounding box'][3] - o['bounding box'][1]) > 5)
    n_degen = len(objs) - n_valid
    
    axes[ax_idx].imshow(vis_rgb)
    axes[ax_idx].set_title(f"{entry['External ID']} — {n_valid} teeth ({n_degen} degenerate)", fontsize=11)
    axes[ax_idx].axis('off')

plt.suptitle('Tufts Dental — Tooth Bounding Boxes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Zoom into individual tooth crops

Show what individual tooth crops look like at the raw bbox size (before resizing to 256×256).

In [ ]:
# Pick one image and show individual tooth crops
entry = data[sample_indices[0]]
img_path = resolve_image_path(entry['External ID'], RADIO_DIR)
img = cv2.imread(img_path)
objs = [o for o in entry['Label']['objects']
        if (o['bounding box'][2] - o['bounding box'][0]) > 10
        and (o['bounding box'][3] - o['bounding box'][1]) > 10]

n_show = min(16, len(objs))
cols = 4
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i in range(n_show):
    o = objs[i]
    top, left, bottom, right = o['bounding box']
    
    # Add 10% padding (matching planned crop strategy)
    h, w = bottom - top, right - left
    pad_h, pad_w = int(h * 0.1), int(w * 0.1)
    t = max(0, top - pad_h)
    l = max(0, left - pad_w)
    b = min(img.shape[0], bottom + pad_h)
    r = min(img.shape[1], right + pad_w)
    
    crop = img[t:b, l:r]
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(crop_rgb)
    axes[i].set_title(f"Tooth {o['title']} — {crop.shape[0]}×{crop.shape[1]}px", fontsize=10)
    axes[i].axis('off')

for i in range(n_show, len(axes)):
    axes[i].axis('off')

plt.suptitle(f"Individual Tooth Crops — {entry['External ID']} (with 10% padding)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Degenerate bbox analysis

Examine the 102 degenerate bounding boxes (h or w ≤ 5px).

In [ ]:
degen_boxes = []
for entry in data:
    for o in entry['Label']['objects']:
        top, left, bottom, right = o['bounding box']
        h = bottom - top
        w = right - left
        if h <= 5 or w <= 5:
            degen_boxes.append({
                'image': entry['External ID'],
                'tooth': o['title'],
                'bbox': o['bounding box'],
                'h': h, 'w': w
            })

print(f"Total degenerate boxes: {len(degen_boxes)}")
print(f"\nBreakdown by size:")
print(f"  h<=0 or w<=0: {sum(1 for d in degen_boxes if d['h'] <= 0 or d['w'] <= 0)}")
print(f"  1x1 (point): {sum(1 for d in degen_boxes if d['h'] == 1 and d['w'] == 1)}")
print(f"  h in 2-5 or w in 2-5: {sum(1 for d in degen_boxes if (1 < d['h'] <= 5 or 1 < d['w'] <= 5) and d['h'] > 0 and d['w'] > 0)}")
print(f"\nSample degenerate boxes:")
for d in degen_boxes[:10]:
    print(f"  {d['image']} tooth {d['tooth']}: bbox={d['bbox']}, size={d['h']}×{d['w']}")

# Which tooth IDs are most commonly degenerate?
degen_ids = Counter(d['tooth'] for d in degen_boxes)
print(f"\nMost commonly degenerate tooth IDs:")
for tid, count in degen_ids.most_common(10):
    print(f"  Tooth {tid}: {count} degenerate boxes")

## 8. YOLO format conversion preview

Show what the YOLO-format labels will look like for a sample image.

In [ ]:
# Preview YOLO format conversion for one image
IMG_H, IMG_W = 840, 1615  # verified consistent
MIN_BBOX_SIZE = 10

entry = data[0]
print(f"Image: {entry['External ID']}")
print(f"\nYOLO format labels (class x_center y_center width height):")
print("-" * 60)

kept = 0
filtered = 0
for o in entry['Label']['objects']:
    top, left, bottom, right = o['bounding box']
    h = bottom - top
    w = right - left
    
    if h <= MIN_BBOX_SIZE or w <= MIN_BBOX_SIZE:
        print(f"  FILTERED tooth {o['title']}: {h}×{w}px (too small)")
        filtered += 1
        continue
    
    x_center = (left + right) / 2.0 / IMG_W
    y_center = (top + bottom) / 2.0 / IMG_H
    norm_w = w / IMG_W
    norm_h = h / IMG_H
    
    print(f"  0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}  "
          f"(tooth {o['title']}, {h}×{w}px)")
    kept += 1

print(f"\nKept: {kept}, Filtered: {filtered}")

## 9. Summary statistics

In [ ]:
total_annots = sum(len(e['Label']['objects']) for e in data)
valid_annots = total_annots - degenerate_count

print("=" * 50)
print("TUFTS DENTAL DATASET SUMMARY")
print("=" * 50)
print(f"Images:              {len(data)}")
print(f"Image dimensions:    840 × 1615 (H × W) — all consistent")
print(f"File format:         .JPG (case mismatch with JSON: .jpg)")
print(f"Total annotations:   {total_annots}")
print(f"Valid annotations:   {valid_annots} (after filtering h/w <= 10)")
print(f"Degenerate (≤5px):   {degenerate_count}")
print(f"Avg teeth/image:     {total_annots/len(data):.1f}")
print(f"Bbox format:         [top, left, bottom, right] in absolute pixels")
print(f"Median tooth size:   {np.median(valid_h):.0f}×{np.median(valid_w):.0f} (H×W)")
print(f"Single class:        tooth (for YOLO: class 0)")
print(f"YOLO split plan:     800 train / 200 val (80/20)")
print("=" * 50)